In [ ]:
#Netflix Dataset Analysis

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Set Seaborn style and palette
sns.set_style("whitegrid")
sns.set_palette("viridis")

In [ ]:
# Load dataset
df = pd.read_csv('/content/Netflix Dataset.csv')

# Standardize column names
df.columns = [c.strip() for c in df.columns]
df.head()


In [ ]:
# Check duplicates
print("Duplicate rows before cleaning:", df.duplicated().sum())

# Drop duplicates
df.drop_duplicates(inplace=True)
print("Duplicate rows after cleaning:", df.duplicated().sum())

# Fill missing values
df['Country'].fillna('Unknown', inplace=True)
df['Category'].fillna('Unknown', inplace=True)
df['Type'].fillna('', inplace=True)


In [ ]:
# Clean 'Category' column
df['type_clean'] = df['Category'].astype(str).str.strip().str.title()

# Split genres and countries into lists
df['genres_list'] = df['Type'].astype(str).str.split(",").apply(lambda lst: [x.strip() for x in lst if x.strip() != ''])
df['country_list'] = df['Country'].astype(str).str.split(",").apply(lambda lst: [x.strip() for x in lst if x.strip() != ''])

# Extract release year
df['release_year'] = pd.to_numeric(df['Release_Date'].astype(str).str.extract(r'(\d{4})', expand=False), errors='coerce').astype('Int64')
dur = df['Duration'].astype(str).fillna("")

In [ ]:


# Extract minutes and seasons
minutes = dur.str.extract(r'(\d+)\s*min', expand=False)
seasons = dur.str.extract(r'(\d+)\s*Season', expand=False)

df['duration_minutes'] = pd.NA
df['seasons'] = pd.NA
df.loc[minutes.notna(), 'duration_minutes'] = pd.to_numeric(minutes[minutes.notna()], errors='coerce').astype('Int64')
df.loc[seasons.notna(), 'seasons'] = pd.to_numeric(seasons[seasons.notna()], errors='coerce').astype('Int64')


In [ ]:
# Explode genres
genres_exploded = df.explode('genres_list')
genres_exploded = genres_exploded[genres_exploded['genres_list'].notna() & (genres_exploded['genres_list'] != "")]
genres_exploded['genre'] = genres_exploded['genres_list'].astype(str)

# Explode countries
countries_exploded = df.explode('country_list')
countries_exploded = countries_exploded[countries_exploded['country_list'].notna() & (countries_exploded['country_list'] != "")]
countries_exploded['country'] = countries_exploded['country_list'].astype(str)


In [ ]:
# Aggregations
type_year = df.groupby(['release_year', 'type_clean']).size().unstack(fill_value=0).sort_index()
top_genres = genres_exploded['genre'].value_counts().nlargest(20)
top_countries = countries_exploded['country'].value_counts().nlargest(20)

print("Row count:", df.shape[0])
print("\nTop genres (top 10):")
print(top_genres.head(10))
print("\nTop countries (top 10):")
print(top_countries.head(10))


In [ ]:
# Line plot: Number of movies vs TV shows per year
plt.figure(figsize=(12,6))
if not type_year.empty:
    type_year_reset = type_year.reset_index().melt(id_vars='release_year', value_name='count', var_name='Type')
    sns.lineplot(data=type_year_reset, x='release_year', y='count', hue='Type', marker="o")
    plt.xlabel("Release Year")
    plt.ylabel("Number of Titles")
    plt.title("Content Count by Type per Year")
    plt.legend(title="Type")
    plt.tight_layout()
    plt.show()


In [ ]:
# Bar plot: Top 20 countries by number of titles
plt.figure(figsize=(12,6))
if not top_countries.empty:
    sns.barplot(x=top_countries.values, y=top_countries.index, orient='h')
    plt.xlabel("Number of Titles")
    plt.ylabel("Country")
    plt.title("Top 20 Countries by Number of Titles")
    plt.tight_layout()
    plt.show()


In [ ]:
# Bar plot: Top 20 genres by frequency
plt.figure(figsize=(12,6))
if not top_genres.empty:
    sns.barplot(x=top_genres.values, y=top_genres.index, orient='h')
    plt.xlabel("Number of Titles")
    plt.ylabel("Genre")
    plt.title("Top 20 Genres by Frequency")
    plt.tight_layout()
    plt.show()


In [ ]:
# Line plot: Trends for top 8 genres over years
top_genre_list = top_genres.index.tolist()[:8]
genre_year = genres_exploded.copy()
genre_year = genre_year[genre_year['genre'].isin(top_genre_list)]

if not genre_year.empty:
    # Group by year and genre
    genre_year = genre_year.groupby(['release_year','genre']).size().unstack(fill_value=0)
    
    # Reindex to include all years
    try:
        year_min = int(df['release_year'].min())
        year_max = int(df['release_year'].max())
        idx = range(year_min, year_max + 1)
        genre_year = genre_year.reindex(idx, fill_value=0)
    except Exception:
        pass
    
    # Plot each genre as a line
    plt.figure(figsize=(12,6))
    for genre in genre_year.columns:
        sns.lineplot(x=genre_year.index, y=genre_year[genre], label=genre)
    plt.xlabel("Release Year")
    plt.ylabel("Number of Titles")
    plt.title("Genre Trends Over Years (Top 8 Genres)")
    plt.legend(title="Genre")
    plt.tight_layout()
    plt.show()
